# PROTOTYPE — Qwen3-VL F0 fallback contract qualification

**Question:** can the exact Qwen3-VL-8B-Instruct-FP8 Kaggle mount and pinned vLLM runtime cold-install without network access, become ready, accept the primary route's exact visual request under its served-model alias, emit one parsed native `take_action(LEFT)` call, and release the process group and GPU memory on shutdown?

This notebook performs one model request. It grants the fallback deployment qualification only—no Tycho behavioral authority, ARC action, score, or optimization.

In [ ]:
import base64
from pathlib import Path

Path('/tmp/PROTOTYPE-qwen3_vl_f0_probe.py').write_bytes(
    base64.b64decode('...')
)


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(8 * 1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

bootstrap_started = time.monotonic()
bootstrap_dir = Path('/kaggle/working/qwen3-vl-f0-bootstrap')
bootstrap_dir.mkdir(parents=True, exist_ok=False)
bootstrap_result = {
    'started_at': datetime.now(UTC).isoformat(),
    'status': 'started',
}
try:
    input_root = Path('/kaggle/input')
    wheel_paths = list(input_root.rglob('vllm-0.28.0-cp38-abi3-manylinux_2_28_x86_64.whl'))
    lock_paths = [
        path
        for path in input_root.rglob('requirements.lock')
        if sha256_file(path) == 'd473b6e092e01f66313e12069ac8873f2ff1c6c3b84b51c1b37312229de5475c'
    ]
    if len(wheel_paths) != 1 or len(lock_paths) != 1:
        raise RuntimeError(
            'expected one pinned vLLM wheel and lockfile in attached inputs, '
            f'found wheels={wheel_paths}, locks={lock_paths}'
        )
    wheel_path = wheel_paths[0]
    lock_path = lock_paths[0]
    actual_wheel_sha256 = sha256_file(wheel_path)
    if actual_wheel_sha256 != 'addb0ffdaafd8155d75e9b3f5ddb3da28fdee9e8a7097ede91f7db2e9e1a3889':
        raise RuntimeError('mounted vLLM wheel does not match the pinned bytes')
    wheelhouse = wheel_path.parent
    runtime_provenance = Path('/tmp/vllm-runtime-provenance.json')
    runtime_provenance.write_text(json.dumps({
        'dataset_source': 'mahdiseddigh/kaggle-t4-stack-2026-08-28',
        'dataset_version': 1,
        'artifact_path': str(wheel_path),
        'artifact_sha256': actual_wheel_sha256,
        'lock_path': str(lock_path),
        'lock_sha256': 'd473b6e092e01f66313e12069ac8873f2ff1c6c3b84b51c1b37312229de5475c',
    }, indent=2) + '\n')

    overrides_path = Path('/tmp/vllm-runtime-overrides.txt')
    overrides_path.write_text('flashinfer-python==0.6.17\n')
    uv_install_command = [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--no-index',
        '--find-links',
        str(wheelhouse),
        'uv==0.12.7',
    ]
    with (bootstrap_dir / 'runtime-install.log').open('w') as install_log:
        subprocess.run(
            uv_install_command,
            check=True,
            stdout=install_log,
            stderr=subprocess.STDOUT,
        )
    uv_executable = shutil.which('uv')
    if not uv_executable:
        raise RuntimeError('uv executable missing after offline installation')
    install_command = [
        uv_executable,
        'pip',
        'install',
        '--system',
        '--no-index',
        '--find-links',
        str(wheelhouse),
        '--override',
        str(overrides_path),
        '--upgrade',
        str(wheel_path),
        'torch==2.13.0+cu130',
        'transformers==5.12.1',
        'flashinfer-python==0.6.17',
        'flash-attn-4==4.0.0b19',
        'triton==3.7.1',
    ]
    (bootstrap_dir / 'runtime-install-command.json').write_text(
        json.dumps({
            'uv_install': uv_install_command,
            'runtime_install': install_command,
            'overrides': overrides_path.read_text().splitlines(),
        }, indent=2) + '\n'
    )
    with (bootstrap_dir / 'runtime-install.log').open('a') as install_log:
        subprocess.run(
            install_command,
            check=True,
            stdout=install_log,
            stderr=subprocess.STDOUT,
        )
    (bootstrap_dir / 'runtime-provenance.json').write_text(
        runtime_provenance.read_text()
    )
    with (bootstrap_dir / 'runtime-packages.json').open('w') as package_log:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'list', '--format', 'json'],
            check=True,
            stdout=package_log,
        )
    bootstrap_result['status'] = 'passed'
except Exception as error:
    bootstrap_result |= {
        'status': 'failed',
        'error_type': type(error).__name__,
        'error': str(error),
    }
    raise
finally:
    bootstrap_result |= {
        'ended_at': datetime.now(UTC).isoformat(),
        'elapsed_s': round(time.monotonic() - bootstrap_started, 3),
    }
    (bootstrap_dir / 'bootstrap-result.json').write_text(
        json.dumps(bootstrap_result, indent=2) + '\n'
    )


In [ ]:
import subprocess
import sys
from pathlib import Path

output_dir = Path('/kaggle/working/qwen3-vl-f0-probe')
probe_command = [
    sys.executable,
    '/tmp/PROTOTYPE-qwen3_vl_f0_probe.py',
    '--output-dir',
    str(output_dir),
    '--input-root',
    '/kaggle/input',
    '--runtime-provenance',
    '/tmp/vllm-runtime-provenance.json',
    '--startup-timeout-s',
    '1200',
    '--request-timeout-s',
    '180',
]
subprocess.run(probe_command, check=True)
print((output_dir / 'result.json').read_text())
